# Kansas Soil Water Atlas: figures

Makes every figure of the atlas (`img/`) and the drought-event table (`files/drought_events.csv`). The page itself (`index.html`) and the other download files are made by `page.ipynb`, which runs in seconds.

**Yearly update:**
1. after a model rerun, run `build_cube.ipynb` (about 9 min);
2. run this notebook (about 7 min);
3. run `page.ipynb` (seconds);
4. commit `index.html`, `img/` and `files/` in kansas-soil-water-atlas.

**Text changes only:** edit `page.ipynb` (chapter text and captions) or `template.html` (intro, about, layout), then run `page.ipynb`.

In [ ]:
import os, json, shutil, time
ROOT = r'D:\OneDrive - Kansas State University\Research projects\kansas-1km'   # the model output (faw/, soil/, boundaries/) is read from here
SEC  = r'D:\OneDrive - Kansas State University\Research projects\kansas-soil-water-atlas'   # this repository: page, figures, code and data
DATA, IMG, FILES = (os.path.join(SEC, d) for d in ('data', 'img', 'files'))
os.chdir(ROOT); os.makedirs(IMG, exist_ok=True); os.makedirs(FILES, exist_ok=True)
import numpy as np, pandas as pd, geopandas as gpd, rasterio
import matplotlib.pyplot as plt, matplotlib.dates as mdates
from matplotlib.colors import BoundaryNorm, to_rgba
import warnings; warnings.filterwarnings('ignore')

REF, DPI = 0.5, 170
INK, TEXT, MUTED, GRID, AXIS = '#121212', '#333333', '#727272', '#e6e6e6', '#9a9a9a'
DRY, WET = '#c8783a', '#2c7fb8'
plt.rcParams.update({'font.family': ['Arial', 'DejaVu Sans'], 'font.size': 9.5, 'axes.edgecolor': AXIS, 'axes.linewidth': 0.8,
                     'axes.labelcolor': TEXT, 'xtick.color': TEXT, 'ytick.color': TEXT, 'xtick.major.size': 3, 'ytick.major.size': 3,
                     'legend.frameon': False,
                     'figure.facecolor': 'white', 'savefig.facecolor': 'white'})
MONTHS = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
MONTHS_LONG = ['January', 'February', 'March', 'April', 'May', 'June', 'July', 'August', 'September', 'October', 'November', 'December']
MSTART = pd.date_range('2001-01-01', periods=12, freq='MS').dayofyear.to_numpy()
TWO_PI = 2 * np.pi

def box(ax):
    "Publication style for every line, bar and scatter chart: full frame and a subtle dotted grid."
    for s in ax.spines.values(): s.set_visible(True); s.set_color(AXIS); s.set_linewidth(0.8)
    ax.grid(color='#cfcfcf', lw=0.6, ls=(0, (1, 2))); ax.set_axisbelow(True)

def save(fig, name):
    "Save to img/ (retry while OneDrive holds the file) and close."
    p = os.path.join(IMG, f'{name}.png')
    for k in range(20):
        try: fig.savefig(p, dpi=DPI, bbox_inches='tight', pad_inches=0.06); break
        except OSError:
            if k == 19: raise
            time.sleep(3)
    plt.close(fig)
    print(f'{name}.png  {os.path.getsize(p) / 1e3:.0f} kB')

def nice_date(d):
    "AP-style date, e.g. Oct. 24, 2022."
    m = ['Jan.', 'Feb.', 'March', 'April', 'May', 'June', 'July', 'Aug.', 'Sept.', 'Oct.', 'Nov.', 'Dec.'][d.month - 1]
    return f'{m} {d.day}, {d.year}'

wk    = pd.read_csv(os.path.join(DATA, 'weeks.csv'), parse_dates=['start', 'end'])
cube  = np.load(os.path.join(DATA, 'faw_weekly_uint8.npy'), mmap_mode='r')
daily = pd.read_csv(os.path.join(DATA, 'statewide_daily.csv'), parse_dates=['date'])
fd    = np.load(os.path.join(DATA, 'faw_days_yearly.npz'))
valid = fd['valid']
with rasterio.open('faw/faw_20020101.tif') as src: profile, shape, transform = src.profile, src.shape, src.transform
lat = transform.f + transform.e * (np.arange(shape[0]) + 0.5)
lon = transform.c + transform.a * (np.arange(shape[1]) + 0.5)
area = np.cos(np.deg2rad(np.broadcast_to(lat[:, None], shape)))[valid] * (abs(transform.a) * 111.32) * (abs(transform.e) * 111.32)
wa = area / area.sum()
with rasterio.open('soil/taw.tif') as src: taw = src.read(1)[valid]
CAPACITY = float(np.sum(taw * area) * 1e-6)                     # km3 when FAW = 1 everywhere
MAF = 0.810714                                                 # million acre-feet per km3

years = np.arange(wk.year.min(), wk.year.max() + 1)
CY = np.array([y for y in years if (wk.year == y).sum() == 52])  # complete calendar years
LATEST, LAST_DAY = int(years[-1]), daily.date.iloc[-1]
daily['year'] = daily.date.dt.year
daily['frac'] = (daily.date.dt.dayofyear - 1) / (365 + daily.date.dt.is_leap_year)
annual = daily.groupby('year')[['deficit', 'surplus']].sum()
annual['complete'] = annual.index.isin(CY)
annual['median_faw'] = daily.groupby('year').p50.median()
key = daily.date.dt.year * 100 + np.minimum((daily.date.dt.dayofyear - 1) // 7 + 1, 52)   # the cube's weeks
print(f'{len(wk)} weeks, {years[0]}-{LATEST} ({len(CY)} complete years), last day {LAST_DAY:%Y-%m-%d}; capacity {CAPACITY:.1f} km3')

## Weekly statewide numbers and map helpers

In [ ]:
med = np.full((len(years), 52), np.nan)
for i, (y, w) in enumerate(zip(wk.year, wk.week)):
    h = np.bincount(cube[i][valid], weights=wa, minlength=101)
    med[y - years[0], w - 1] = np.searchsorted(np.cumsum(h), 0.5) / 100
WD = np.full((len(years), 52), np.nan); WS = WD.copy()
for k, g in daily.groupby(key)[['deficit', 'surplus']].sum().reindex(wk.year * 100 + wk.week).iterrows():
    WD[k // 100 - years[0], k % 100 - 1], WS[k // 100 - years[0], k % 100 - 1] = g.deficit, g.surplus

from rasterio.warp import reproject, calculate_default_transform, Resampling
from scipy.ndimage import distance_transform_edt
from scipy.signal import savgol_filter
PLOT_CRS = '+proj=aea +lat_1=37.5 +lat_2=39.5 +lat_0=38.5 +lon_0=-98.3 +datum=NAD83 +units=m +no_defs'
dst_transform, dst_w, dst_h = calculate_default_transform(profile['crs'], PLOT_CRS, shape[1], shape[0],
                                                          *rasterio.transform.array_bounds(*shape, transform), resolution=1000)
extent = [dst_transform.c, dst_transform.c + dst_transform.a * dst_w, dst_transform.f + dst_transform.e * dst_h, dst_transform.f]
px = dst_transform.c + dst_transform.a * (np.arange(dst_w) + 0.5); py = dst_transform.f + dst_transform.e * (np.arange(dst_h) + 0.5)
kansas = gpd.read_file('boundaries/kansas_state_bnd.geojson').explode(index_parts=False)
kansas = kansas[kansas.geom_type.str.endswith('Polygon')]
counties = gpd.read_file('boundaries/kansas_county_bnd.geojson')
kansas_p, counties_p = kansas.to_crs(PLOT_CRS), counties.to_crs(PLOT_CRS)

def grid_of(v):
    g = np.full(shape, np.nan, 'float32'); g[valid] = v; return g
def to_plot(g):
    out = np.full((dst_h, dst_w), np.nan, 'float32')
    reproject(g.astype('float32'), out, src_transform=transform, src_crs=profile['crs'], dst_transform=dst_transform,
              dst_crs=PLOT_CRS, src_nodata=np.nan, dst_nodata=np.nan, resampling=Resampling.nearest)
    return out
def smooth(v, n=15):
    "Savitzky-Golay smoothing for the long-term maps (gaps filled from the nearest pixel first)."
    g = grid_of(v); m = np.isfinite(g)
    g = g[tuple(distance_transform_edt(~m, return_distances=False, return_indices=True))]
    g = savgol_filter(savgol_filter(g, n, 2, axis=0, mode='nearest'), n, 2, axis=1, mode='nearest')
    g = np.clip(g, np.nanmin(v), np.nanmax(v)); g[~m] = np.nan; return g
def outline(ax, lw_c=0.3, lw_s=0.7):
    counties_p.boundary.plot(ax=ax, color='white', lw=lw_c, alpha=0.75); kansas_p.boundary.plot(ax=ax, color='#444', lw=lw_s)
    ax.set_aspect('equal'); ax.set_xlim(extent[0], extent[1]); ax.set_ylim(extent[2], extent[3]); ax.axis('off')
def hbar(fig, ax, im, label, ticks=None, width=0.5):
    "Small horizontal colour bar centred under a map."
    cax = ax.inset_axes([(1 - width) / 2, -0.10, width, 0.05])
    cb = fig.colorbar(im, cax=cax, orientation='horizontal', ticks=ticks); cb.outline.set_visible(False)
    cb.ax.tick_params(labelsize=8, length=2); cb.set_label(label, fontsize=8.5, color=TEXT); return cb
FAW_CMAP = plt.get_cmap('Spectral')
print('ok')

## 1 · Twenty-five years at a glance

In [ ]:
def week_axis(ax):
    ax.set_xticks((MSTART - 1) / 7 + 0.5); ax.set_xticklabels(MONTHS, ha='left')

def heat(ax, M, cmap, vmin, vmax, show_years=True):
    cm = plt.get_cmap(cmap).copy(); cm.set_bad('white')
    im = ax.pcolormesh(np.arange(0.5, 53), np.arange(years[0] - 0.5, years[-1] + 1), np.ma.masked_invalid(M), cmap=cm, vmin=vmin, vmax=vmax,
                       edgecolors='white', linewidth=0.5)
    ax.set_ylim(years[-1] + 0.5, years[0] - 0.5); ax.set_xlim(0.5, 52.5); week_axis(ax)
    ax.set_yticks(years); ax.set_yticklabels([str(y) if show_years else '' for y in years], fontsize=8)
    for s in ax.spines.values(): s.set_visible(False)
    ax.tick_params(length=0)
    return im

def top_bar(fig, ax, im, label, ticks, y=1.035):
    cax = ax.inset_axes([0.0, y, 0.32, 0.018])
    cb = fig.colorbar(im, cax=cax, orientation='horizontal', ticks=ticks); cb.outline.set_visible(False)
    cb.ax.tick_params(labelsize=8, length=2); cb.ax.xaxis.set_ticks_position('bottom')
    cax.set_title(label, fontsize=9, color=TEXT, loc='left', pad=3)

fig, ax = plt.subplots(figsize=(10, 7.2))
im = heat(ax, med, 'Spectral', 0, 1)
top_bar(fig, ax, im, 'Statewide median FAW (0 = wilting point, 1 = field capacity)', [0, 0.25, 0.5, 0.75, 1])
save(fig, 'glance_weekly_median')

fig, axs = plt.subplots(1, 2, figsize=(12, 7.2), gridspec_kw=dict(wspace=0.04))
im = heat(axs[0], WD, 'Oranges', 0, 3); top_bar(fig, axs[0], im, 'Deficit-days per week', [0, 1, 2, 3])
im = heat(axs[1], WS, 'Blues', 0, 3, show_years=False); top_bar(fig, axs[1], im, 'Surplus-days per week', [0, 1, 2, 3])
save(fig, 'glance_weekly_faw_days')

fig, ax = plt.subplots(figsize=(10, 4.4))
x = annual.index.to_numpy(); full = annual.complete.to_numpy(); al = np.where(full, 1.0, 0.45)
ax.bar(x, annual.surplus, color=[to_rgba(WET, a) for a in al], width=0.72)
ax.bar(x, -annual.deficit, color=[to_rgba(DRY, a) for a in al], width=0.72)
ax.plot(x, annual.surplus - annual.deficit, color=INK, lw=1, marker='o', ms=3.2, zorder=3)
ax.axhline(0, color=INK, lw=0.8)
ax.set_xticks(x[::2]); ax.set_xlim(x[0] - 0.7, x[-1] + 0.7); box(ax)
lim = np.ceil(max(annual.deficit.max(), annual.surplus.max()) / 20) * 20 + 10
ax.set_ylim(-lim, lim); ax.set_yticks(np.arange(-lim + 10, lim, 20)); ax.set_yticklabels([f'{abs(v):.0f}' for v in ax.get_yticks()])
ax.text(x[0] - 0.5, lim - 4, 'Surplus-days', color=WET, fontsize=9.5, fontweight='bold', va='top')
ax.text(x[0] - 0.5, -lim + 4, 'Deficit-days', color=DRY, fontsize=9.5, fontweight='bold', va='bottom')
yD, yS = int(annual[annual.complete].deficit.idxmax()), int(annual[annual.complete].surplus.idxmax())
ax.annotate(f'{yD}: {annual.deficit[yD]:.0f} deficit-days', (yD, -annual.deficit[yD]), xytext=(0, -12), textcoords='offset points', ha='center', va='top', fontsize=8.5, color=TEXT)
ax.annotate(f'{yS}: {annual.surplus[yS]:.0f} surplus-days', (yS, annual.surplus[yS]), xytext=(0, 6), textcoords='offset points', ha='center', fontsize=8.5, color=TEXT)
ax.annotate('Net', (x[-2], (annual.surplus - annual.deficit).iloc[-2]), xytext=(8, 8), textcoords='offset points', fontsize=8.5, color=INK)
if not full[-1]: ax.annotate(f'Through\n{LAST_DAY:%b} {LAST_DAY.day}', (x[-1], -annual.deficit.iloc[-1]), xytext=(0, -6), textcoords='offset points', ha='center', va='top', fontsize=7.5, color=MUTED)
ax.set_ylabel('FAW-days per year')
save(fig, 'glance_annual_ledger')

## 2 · The soil reservoir

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.9))
ax.fill_between(daily.date, 0, daily.paw_km3, color=WET, alpha=0.16, lw=0)
ax.plot(daily.date, daily.paw_km3, color=WET, lw=0.7)
ax.axhline(CAPACITY, color=AXIS, lw=0.8)
ax.text(daily.date.iloc[30], CAPACITY + 0.4, f'Field capacity: {CAPACITY:.1f} km³ ({CAPACITY * MAF:.1f} million acre-feet)', fontsize=8.5, color=TEXT, va='bottom')
i0, i1 = daily.paw_km3.idxmin(), daily.paw_km3.idxmax()
for i, dx, ha in [(i0, 7, 'left'), (i1, -7, 'right')]:
    ax.plot(daily.date[i], daily.paw_km3[i], 'o', color=INK, ms=3.5)
    ax.annotate(f'{daily.paw_km3[i]:.1f} km³, {nice_date(daily.date[i])}', (daily.date[i], daily.paw_km3[i]), xytext=(dx, 0), textcoords='offset points', fontsize=8.5, color=INK, va='center', ha=ha)
ax.set_ylim(0, CAPACITY * 1.12); ax.set_xlim(daily.date.iloc[0], daily.date.iloc[-1])
ax.xaxis.set_major_locator(mdates.YearLocator(2)); ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.set_ylabel('Plant available water stored (km³)'); box(ax)
sec = ax.secondary_yaxis('right', functions=(lambda v: v * MAF, lambda v: v / MAF)); sec.set_ylabel('Million acre-feet'); sec.spines['right'].set_color(AXIS)
save(fig, 'reservoir_timeseries')

wkr = daily.groupby(key).paw_km3.mean().reindex(wk.year * 100 + wk.week).to_numpy()
sel = [int(np.argmin(wkr)), int(np.argmax(wkr))]
ZMAX = float(np.ceil(np.percentile(taw, 99.5) / 10) * 10)
lev = np.arange(0, ZMAX + 1, 10); norm = BoundaryNorm(lev, 256)
fig, axs = plt.subplots(1, 2, figsize=(9, 2.6), gridspec_kw=dict(wspace=0.06))
for ax, i, lab in zip(axs, sel, ['The driest week', 'The wettest week']):
    stored = cube[i][valid].astype('float32') / 100 * taw
    im = ax.imshow(to_plot(grid_of(stored)), extent=extent, cmap='Blues', norm=norm, interpolation='nearest')
    outline(ax)
    ax.set_title(f'{lab}\n', fontsize=10.5, fontweight='bold', loc='left', color=INK)
    ax.text(0, 1.03, f'Week of {nice_date(wk.start[i])} · {wkr[i]:.1f} km³, {wkr[i] / CAPACITY:.0%} of capacity', transform=ax.transAxes, fontsize=8.5, color=TEXT)
cax = fig.add_axes([0.3, 0.0, 0.4, 0.04])
cb = fig.colorbar(im, cax=cax, orientation='horizontal', ticks=np.arange(0, ZMAX + 1, 30)); cb.outline.set_visible(False)
cb.ax.tick_params(labelsize=8, length=2); cb.set_label('Plant available water stored in the top 60 cm (mm)', fontsize=8.5, color=TEXT)
save(fig, 'reservoir_where')

## 3 · Every year since 2002

In [ ]:
def cycle(ax, y, size='small'):
    d = daily[daily.year == y]
    t, r = TWO_PI * d.frac.to_numpy(), d.p50.to_numpy()
    ax.set_theta_zero_location('N'); ax.set_theta_direction(-1)
    ax.fill_between(t, r, REF, where=r < REF, color=DRY, alpha=0.75, lw=0, interpolate=True, zorder=3)
    ax.fill_between(t, r, REF, where=r > REF, color=WET, alpha=0.65, lw=0, interpolate=True, zorder=3)
    ax.plot(t, r, color=INK, lw=0.7 if size == 'small' else 1.1, zorder=4)
    ax.set_ylim(0, 1); ax.set_yticks([0.25, 0.5, 0.75])
    ax.set_yticklabels([])
    for v in (0, 0.5, 1):                                       # FAW scale along the January line
        ax.annotate(f'{v:g}', (0, v), xytext=(2, -1 if v == 1 else 0), textcoords='offset points', ha='left', va='top' if v == 1 else 'center',
                    fontsize=5.5 if size == 'small' else 7, color=MUTED, zorder=5)
    ax.set_xticks(TWO_PI * (MSTART - 1) / 365)
    ax.set_xticklabels([m[0] for m in MONTHS] if size == 'small' else MONTHS, fontsize=6.5 if size == 'small' else 8, color=MUTED)
    ax.tick_params(pad=-2 if size == 'small' else 1); ax.grid(color=GRID, lw=0.6); ax.spines['polar'].set_color(AXIS); ax.spines['polar'].set_linewidth(0.6)

nc = 5; nr = -(-len(years) // nc)
fig, axs = plt.subplots(nr, nc, figsize=(10, 2.3 * nr), subplot_kw=dict(projection='polar'))
for ax, y in zip(axs.flat, years):
    cycle(ax, int(y))
    D, S = annual.loc[y, 'deficit'], annual.loc[y, 'surplus']
    ax.set_title(f'{y}{"" if y in CY else "*"}', fontsize=10, fontweight='bold', color=INK, pad=13)
    ax.text(0.5, 1.13, f'{D:.0f} deficit · {S:.0f} surplus', transform=ax.transAxes, ha='center', fontsize=7.5, color=MUTED)
for ax in axs.flat[len(years):]: ax.axis('off')
plt.tight_layout(h_pad=2.4, w_pad=1.2)
save(fig, 'years_dials')

def race(ax, col, cmap, title):
    comp = annual[annual.complete][col]
    hi = comp.nlargest(3).index.tolist() + [LATEST]
    cs = dict(zip(hi, [plt.get_cmap(cmap)(v) for v in (0.95, 0.75, 0.55)] + [INK]))
    for y, g in daily.groupby('year'):
        if y not in hi: ax.plot(g.date.dt.dayofyear, g[col].cumsum(), color='#cfcfcf', lw=0.7, zorder=2)
    for y in hi:
        g = daily[daily.year == y]; xx, yy = g.date.dt.dayofyear.to_numpy(), g[col].cumsum().to_numpy()
        ax.plot(xx, yy, color=cs[y], lw=1.8 if y == LATEST else 1.4, zorder=3)
        ax.text(xx[-1] + 3, yy[-1], str(y), color=cs[y], fontsize=8.5, fontweight='bold', va='center')
    ax.set_xlim(1, 380); ax.set_xticks(MSTART); ax.set_xticklabels(MONTHS, ha='left'); ax.set_ylim(0, None)
    ax.set_title(title, loc='left', fontsize=10.5, fontweight='bold', color=INK); box(ax)

fig, axs = plt.subplots(1, 2, figsize=(12, 4.1), gridspec_kw=dict(wspace=0.18))
race(axs[0], 'deficit', 'YlOrRd', 'Deficit-days, running total'); race(axs[1], 'surplus', 'YlGnBu', 'Surplus-days, running total')
save(fig, 'years_running_totals')

fig, ax = plt.subplots(figsize=(6.6, 6.6), subplot_kw=dict(projection='polar'))
ax.set_theta_zero_location('N'); ax.set_theta_direction(-1)
HOLE = 8
TH, RR = np.meshgrid(TWO_PI * np.arange(53) / 52, HOLE + np.arange(len(years) + 1))
im = ax.pcolormesh(TH, RR, np.ma.masked_invalid(med), cmap='Spectral', vmin=0, vmax=1, shading='flat', edgecolors='white', linewidth=0.15)
ax.set_ylim(0, HOLE + len(years)); ax.set_yticks([])
ax.set_xticks(TWO_PI * (MSTART - 1) / 365); ax.set_xticklabels(MONTHS, color=MUTED); ax.grid(False); ax.spines['polar'].set_visible(False)
for k in range(0, len(years), 4):
    ax.text(TWO_PI * 0.004, HOLE + k + 0.5, str(years[k]), fontsize=7, ha='left', va='center', color=INK,
            bbox=dict(boxstyle='round,pad=0.1', fc='white', ec='none', alpha=0.8))
cax = ax.inset_axes([0.3, -0.08, 0.4, 0.02])
cb = fig.colorbar(im, cax=cax, orientation='horizontal', ticks=[0, 0.5, 1]); cb.outline.set_visible(False); cb.ax.tick_params(labelsize=8, length=2)
cb.set_label('Statewide median FAW', fontsize=8.5, color=TEXT)
save(fig, 'years_rings')

## 4 · Across the state

In [ ]:
full_i = np.isin(fd['years'], CY)
Dm, Sm = np.median(fd['deficit'][full_i], 0), np.median(fd['surplus'][full_i], 0)
hi = float(np.ceil(np.percentile(np.r_[Dm, Sm], 99) / 10) * 10)
fig, axs = plt.subplots(1, 2, figsize=(12, 3.6), gridspec_kw=dict(wspace=0.06))
for ax, v, cm, t in zip(axs, [Dm, Sm], ['YlOrRd', 'YlGnBu'], ['Deficit-days in a typical year', 'Surplus-days in a typical year']):
    im = ax.contourf(px, py, to_plot(smooth(v)), levels=np.arange(0, hi + 1, 10), cmap=cm, extend='max')
    outline(ax); ax.set_title(t, loc='left', fontsize=10.5, fontweight='bold', color=INK)
    hbar(fig, ax, im, 'FAW-days per year (median of 2002–2025)', ticks=np.arange(0, hi + 1, 20))
save(fig, 'space_typical_year')

colsum = valid.sum(0)
H = np.empty((len(wk), shape[1]), 'float32')
for i in range(len(wk)):
    g = cube[i].astype('float32'); g[~valid] = 0
    H[i] = g.sum(0) / np.maximum(colsum, 1) / 100
H[:, colsum == 0] = np.nan
edges_t = np.r_[wk.start.to_numpy(), (wk.end.iloc[-1] + pd.Timedelta(days=1)).to_datetime64()]
edges_x = transform.c + transform.a * np.arange(shape[1] + 1)
fig, ax = plt.subplots(figsize=(10, 3.9))
im = ax.pcolormesh(edges_t, edges_x, np.ma.masked_invalid(H.T), cmap='Spectral', vmin=0, vmax=1, shading='flat', rasterized=True)
ax.set_yticks(np.arange(-102, -94.5, 1)); ax.set_yticklabels([f'{-v:.0f}°W' for v in np.arange(-102, -94.5, 1)], fontsize=8)
ax.xaxis.set_major_locator(mdates.YearLocator(2)); ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
for s in ax.spines.values(): s.set_visible(False)
ax.text(1.005, 0.02, 'West', transform=ax.transAxes, fontsize=8.5, color=MUTED, va='bottom'); ax.text(1.005, 0.98, 'East', transform=ax.transAxes, fontsize=8.5, color=MUTED, va='top')
top_bar(fig, ax, im, 'Mean FAW of each north–south strip', [0, 0.25, 0.5, 0.75, 1], y=1.09)
save(fig, 'space_west_east')

cls = [f'c{i}' for i in range(10)]
CW = daily.groupby(key)[cls].mean().reindex(wk.year * 100 + wk.week).to_numpy() * 100
colors = plt.get_cmap('Spectral')(np.linspace(0.03, 0.97, 10))
fig, ax = plt.subplots(figsize=(10, 3.6))
ax.stackplot(wk.start + pd.Timedelta(days=3), *CW.T, colors=colors, lw=0, labels=[f'{i / 10:.1f}–{(i + 1) / 10:.1f}' for i in range(10)])
ax.set_ylim(0, 100); ax.set_xlim(wk.start.iloc[0], wk.end.iloc[-1]); ax.set_ylabel('Share of Kansas (%)')
ax.xaxis.set_major_locator(mdates.YearLocator(2)); ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.axhline(50, color='white', lw=0.6); box(ax)
ax.legend(title='FAW', fontsize=7.5, title_fontsize=8, loc='center left', bbox_to_anchor=(1.01, 0.5), reverse=True)
save(fig, 'space_area_classes')

## 5 · Droughts

Connected regions of 8-km block-weeks with FAW below `EV_T` in space and time (faces touching). Each event has a start, an end, a peak area and a deficit volume in statewide FAW-days: Σ (0.5 − FAW) × block area × 7 days ÷ area of Kansas. Only events that last at least 4 weeks and reach 5 % of the state are kept.

In [ ]:
from scipy import ndimage
BLK, EV_T, MIN_WEEKS, MIN_AREA = 8, 0.2, 4, 0.05
area_grid = np.where(valid, np.cos(np.deg2rad(np.broadcast_to(lat[:, None], shape))), 0)
blk_area = area_grid.reshape(shape[0] // BLK, BLK, shape[1] // BLK, BLK).sum((1, 3))
def coarse(i0, i1):
    X = np.asarray(cube[i0:i1]).astype('float32') / 100; X[:, ~valid] = np.nan
    with np.errstate(invalid='ignore'):
        return np.nanmean(X.reshape(X.shape[0], shape[0] // BLK, BLK, shape[1] // BLK, BLK), axis=(2, 4))
Cb = np.concatenate([coarse(i, min(i + 104, len(wk))) for i in range(0, len(wk), 104)])
lab, n = ndimage.label(np.nan_to_num(Cb, nan=1) < EV_T)
tt, rr, cc = np.nonzero(lab)
vox = pd.DataFrame(dict(ev=lab[tt, rr, cc], t=tt, a=blk_area[rr, cc] / blk_area.sum(), d=(REF - Cb[tt, rr, cc]) * blk_area[rr, cc] / blk_area.sum() * 7))
per_t = vox.groupby(['ev', 't']).a.sum()
ev = pd.DataFrame(dict(t0=vox.groupby('ev').t.min(), t1=vox.groupby('ev').t.max(), peak=per_t.groupby('ev').max(),
                       t_peak=per_t.groupby('ev').idxmax().str[1], volume=vox.groupby('ev').d.sum()))
ev['weeks'] = ev.t1 - ev.t0 + 1
ev = ev[(ev.weeks >= MIN_WEEKS) & (ev.peak >= MIN_AREA)].copy()
ev['start'], ev['end'], ev['peak_week'] = wk.start[ev.t0].values, wk.end[ev.t1].values, wk.start[ev.t_peak].values
ev = ev.sort_values('start').reset_index(drop=True)
print(f'{len(ev)} events kept')

fig, ax = plt.subplots(figsize=(10, 4.0))
sc = ax.scatter(ev.start, ev.weeks, s=ev.peak * 1100, c=ev.volume, cmap='YlOrRd', vmin=0, ec=INK, lw=0.5, alpha=0.9, zorder=3)
for _, e in ev.nlargest(6, 'volume').iterrows():
    ax.annotate(f'{e.start:%b %Y}', (e.start, e.weeks), xytext=(0, 3 + np.sqrt(e.peak * 1100) / 2), textcoords='offset points', ha='center', fontsize=8.5, color=INK)
ax.set_ylabel('Duration (weeks)'); ax.set_ylim(0, ev.weeks.max() * 1.3); box(ax)
ax.xaxis.set_major_locator(mdates.YearLocator(2)); ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.set_xlim(wk.start.iloc[0] - pd.Timedelta(days=200), wk.end.iloc[-1] + pd.Timedelta(days=120))
cax = ax.inset_axes([0.70, 1.12, 0.28, 0.025])
cb = fig.colorbar(sc, cax=cax, orientation='horizontal'); cb.outline.set_visible(False); cb.ax.tick_params(labelsize=8, length=2)
cax.set_title('Deficit volume (statewide FAW-days)', fontsize=8.5, color=TEXT, loc='left', pad=3)
for s_ in (0.1, 0.4, 0.7): ax.scatter([], [], s=s_ * 1100, color='none', ec=INK, lw=0.5, label=f'{s_:.0%}')
ax.legend(title='Peak share of Kansas', fontsize=8, title_fontsize=8.5, loc='lower left', ncol=3, labelspacing=1.4, borderpad=0.9, columnspacing=1.6, bbox_to_anchor=(0, 1.01))
save(fig, 'droughts_events')
events = ev[['start', 'end', 'peak_week', 'weeks', 'peak', 'volume']].assign(peak=lambda d: (d.peak * 100).round(1), volume=lambda d: d.volume.round(2))
events.to_csv(os.path.join(FILES, 'drought_events.csv'), index=False, date_format='%Y-%m-%d')
events.nlargest(6, 'volume')

## 6 · One figure per year
Complete calendar years only: a year gets its figure once the model run reaches Dec. 31. Each figure carries its own title, so it stands alone. Statewide dial, maps of the year's deficit-days and surplus-days (same scale every year), and the mean FAW of each month (weeks are assigned to the month of their middle day).

In [ ]:
from matplotlib.gridspec import GridSpec
mid_month = (wk.start + pd.Timedelta(days=3)).dt.month.to_numpy()
YL = np.arange(0, 161, 10); YNORM = BoundaryNorm(YL, 256)
FAW_NORM = BoundaryNorm(np.linspace(0, 1, 21), 256)
yi = {int(y): k for k, y in enumerate(fd['years'])}

def year_figure(y):
    fig = plt.figure(figsize=(12, 6.6))
    gs = GridSpec(3, 6, figure=fig, height_ratios=[2.4, 1, 1], hspace=0.22, wspace=0.04, bottom=0.10, top=0.82)
    fig.text(0.125, 0.985, f'Kansas soil water, {y}', fontsize=16, fontweight='bold', color=INK, va='top')
    fig.text(0.125, 0.945, f'{annual.deficit[y]:.0f} deficit-days and {annual.surplus[y]:.0f} surplus-days statewide · fraction of plant available water (FAW), top 60 cm · Kansas Soil Water Atlas',
             fontsize=9.5, color=MUTED, va='top')
    ax = fig.add_subplot(gs[0, 0:2], projection='polar'); cycle(ax, y, size='big')
    ax.set_title('Statewide median FAW, day by day', fontsize=9.5, color=TEXT, pad=16)
    for k, (col, cmap, t) in enumerate([('deficit', 'Oranges', 'Deficit-days'), ('surplus', 'Blues', 'Surplus-days')]):
        a = fig.add_subplot(gs[0, 2 + 2 * k: 4 + 2 * k])
        im = a.imshow(to_plot(grid_of(fd[col][yi[y]])), extent=extent, cmap=cmap, norm=YNORM, interpolation='nearest')
        outline(a); a.set_title(t, fontsize=10, fontweight='bold', color=INK, loc='left')
        hbar(fig, a, im, 'FAW-days', ticks=[0, 40, 80, 120, 160], width=0.6)
    for m in range(12):
        a = fig.add_subplot(gs[1 + m // 6, m % 6])
        i = np.flatnonzero((wk.year == y).to_numpy() & (mid_month == m + 1))
        X = np.asarray(cube[i]).astype('float32')[:, valid].mean(0) / 100
        a.imshow(to_plot(grid_of(X)), extent=extent, cmap='Spectral', norm=FAW_NORM, interpolation='nearest')
        outline(a, lw_c=0.2, lw_s=0.5)
        a.set_title(MONTHS_LONG[m], fontsize=9, color=TEXT, loc='left', pad=2)
    cax = fig.add_axes([0.38, 0.05, 0.26, 0.016])
    cb = fig.colorbar(plt.cm.ScalarMappable(FAW_NORM, 'Spectral'), cax=cax, orientation='horizontal', ticks=[0, 0.25, 0.5, 0.75, 1])
    cb.outline.set_visible(False); cb.ax.tick_params(labelsize=8, length=2); cb.set_label('Monthly mean FAW', fontsize=8.5, color=TEXT)
    save(fig, f'year_{y}')

t0 = time.time()
for y in CY: year_figure(int(y))
for f in os.listdir(IMG):                                       # a figure left from a year that is no longer complete
    if f.startswith('year_') and f[5:9].isdigit() and int(f[5:9]) not in CY: os.remove(os.path.join(IMG, f))
print(f'{len(CY)} year figures in {time.time() - t0:.0f} s')